## SETUP & DATA LOADING

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import ast
from collections import Counter
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.gridspec import GridSpec

# NLP
import unicodedata
from collections import defaultdict

# Config
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print("✅ Libraries loaded")

# ── LOAD RAW JSON ──────────────────────────────────────────

RAW_PATH = 'BanglishRev.json' # update to your local path

with open(RAW_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

print(f"✅ Loaded {len(raw_data):,} product entries")

## DATA TRANSFORMATION

In [ ]:
def flatten_banglishrev(raw_data: list) -> pd.DataFrame:
    """
    Flatten nested BanglishRev JSON into a review-level DataFrame.
    Each row = one review. Product metadata is broadcast to all its reviews.
    """
    records = []

    for product in raw_data:
        # ── Product-level fields ──────────────────────────
        product_id   = product.get('Product ID')
        category_raw = product.get('Category', [])  # may be list or string
        avg_rating   = product.get('Average Rating')
        score_counts = product.get('Score Counts', {})  # {1:n, 2:n, ...}

        # Normalize category hierarchy → string path
        if isinstance(category_raw, list):
            category_path = ' > '.join([str(c) for c in category_raw if c])
            category_l1   = category_raw[0] if len(category_raw) > 0 else None
            category_l2   = category_raw[1] if len(category_raw) > 1 else None
            category_l3   = category_raw[2] if len(category_raw) > 2 else None
        else:
            category_path = str(category_raw)
            category_l1   = category_raw
            category_l2   = None
            category_l3   = None

        reviews = product.get('Reviews', [])
        if not reviews:  # products with no reviews still tracked
            records.append({
                'product_id': product_id,
                'category_path': category_path,
                'category_l1': category_l1,
                'category_l2': category_l2,
                'category_l3': category_l3,
                'product_avg_rating': avg_rating,
                'score_count_1': score_counts.get('1', 0),
                'score_count_2': score_counts.get('2', 0),
                'score_count_3': score_counts.get('3', 0),
                'score_count_4': score_counts.get('4', 0),
                'score_count_5': score_counts.get('5', 0),
                **{k: None for k in [
                    'buyer_id','rating','review_text','review_date',
                    'date_bought','likes','dislikes','has_reply',
                    'reply_text','has_images','image_count'
                ]}
            })
            continue

        for review in reviews:
            images     = review.get('Images', [])
            img_count  = len(images) if isinstance(images, list) else 0
            reply      = review.get('Reply', None)

            records.append({
                # Product fields
                'product_id'         : product_id,
                'category_path'      : category_path,
                'category_l1'        : category_l1,
                'category_l2'        : category_l2,
                'category_l3'        : category_l3,
                'product_avg_rating' : avg_rating,
                'score_count_1'      : score_counts.get('1', 0),
                'score_count_2'      : score_counts.get('2', 0),
                'score_count_3'      : score_counts.get('3', 0),
                'score_count_4'      : score_counts.get('4', 0),
                'score_count_5'      : score_counts.get('5', 0),

                # Review fields
                'buyer_id'           : review.get('Buyer ID'),
                'rating'             : review.get('Current Rating'),
                'review_text'        : review.get('Review Content'),
                'review_date'        : review.get('Review Date'),
                'date_bought'        : review.get('Date Bought'),
                'likes'              : review.get('Likes', 0),
                'dislikes'           : review.get('Dislikes', 0),
                'has_reply'          : 1 if reply else 0,
                'reply_text'         : reply,
                'has_images'         : 1 if img_count > 0 else 0,
                'image_count'        : img_count,
            })

    df = pd.DataFrame(records)
    return df

df = flatten_banglishrev(raw_data)
print(f"✅ Flattened shape: {df.shape}")
print(f"   Rows (reviews): {len(df):,}")
print(f"   Columns: {df.shape[1]}")

## Building Schema

In [ ]:
SCHEMA = {
    'product_id'         : 'Unique product identifier (item in rec-sys)',
    'category_path'      : 'Full category hierarchy as string',
    'category_l1/l2/l3'  : 'Individual category levels for granular analysis',
    'product_avg_rating' : 'Pre-computed avg rating from product metadata',
    'score_count_1..5'   : 'Pre-computed rating frequency counts per product',
    'buyer_id'           : 'Unique user identifier (user in rec-sys)',
    'rating'             : 'Review star rating 1–5 (interaction signal)',
    'review_text'        : 'Raw review content (NLP input)',
    'review_date'        : 'When review was written',
    'date_bought'        : 'When product was purchased',
    'likes'              : 'Helpful votes on review',
    'dislikes'           : 'Not-helpful votes on review',
    'has_reply'          : 'Binary: seller responded (0/1)',
    'reply_text'         : 'Seller reply text',
    'has_images'         : 'Binary: review has attached images (0/1)',
    'image_count'        : 'Number of images attached',
}
for col, desc in SCHEMA.items():
    print(f"  {col:<25} → {desc}")

## DATA QUALITY ANALYSIS

In [ ]:
print("=" * 65)
print("2.1 MISSING VALUE ANALYSIS")
print("=" * 65)

missing = pd.DataFrame({
    'dtype'     : df.dtypes,
    'missing_n' : df.isnull().sum(),
    'missing_%' : (df.isnull().sum() / len(df) * 100).round(2),
    'unique'    : df.nunique(),
    'sample'    : df.apply(lambda c: c.dropna().iloc[0] if c.dropna().shape[0] > 0 else 'ALL NULL')
})
missing = missing.sort_values('missing_%', ascending=False)
print(missing.to_string())

# ── Type coercions ─────────────────────────────────────────
print("\n" + "=" * 65)
print("2.2 TYPE VALIDATION & COERCION")
print("=" * 65)

df['rating']             = pd.to_numeric(df['rating'], errors='coerce')
df['likes']              = pd.to_numeric(df['likes'], errors='coerce').fillna(0).astype(int)
df['dislikes']           = pd.to_numeric(df['dislikes'], errors='coerce').fillna(0).astype(int)
df['image_count']        = pd.to_numeric(df['image_count'], errors='coerce').fillna(0).astype(int)
df['product_avg_rating'] = pd.to_numeric(df['product_avg_rating'], errors='coerce')

for col in ['review_date', 'date_bought']:
    df[col] = pd.to_datetime(df[col], errors='coerce', infer_datetime_format=True)

invalid_ratings = df[~df['rating'].isin([1,2,3,4,5]) & df['rating'].notna()]
print(f"  Invalid ratings (outside 1–5): {len(invalid_ratings):,}")
print(f"  review_date parse failures:    {df['review_date'].isnull().sum():,}")
print(f"  date_bought parse failures:    {df['date_bought'].isnull().sum():,}")

# ── Duplicate detection ────────────────────────────────────
print("\n" + "=" * 65)
print("2.3 DUPLICATE DETECTION")
print("=" * 65)

dup_exact    = df.duplicated().sum()
dup_user_item= df.duplicated(subset=['buyer_id','product_id']).sum()
dup_text     = df.duplicated(subset=['review_text']).sum()

print(f"  Exact duplicate rows             : {dup_exact:,}")
print(f"  Duplicate (user, product) pairs  : {dup_user_item:,}  ← critical for rec-sys")
print(f"  Duplicate review texts           : {dup_text:,}  ← bot/copy-paste risk")

# Drop exact duplicates; flag user-item dupes for rec-sys
df = df.drop_duplicates()
df['is_dup_interaction'] = df.duplicated(subset=['buyer_id','product_id'], keep=False).astype(int)

# ── Outlier detection ──────────────────────────────────────
print("\n" + "=" * 65)
print("2.4 OUTLIER DETECTION")
print("=" * 65)

for col in ['likes','dislikes','image_count']:
    q99 = df[col].quantile(0.99)
    extreme = df[df[col] > q99]
    print(f"  {col:<15}: 99th pctile = {q99:.0f} | extreme rows = {len(extreme):,}")

# Diagnose negative values — run this before the plot
for col in ['likes', 'dislikes', 'image_count']:
    neg_count = (df[col] < 0).sum()
    print(f"  {col:<15}: negatives = {neg_count:,}  | min = {df[col].min()}")

# ── plotting block ─────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['likes', 'dislikes', 'image_count']):
    # 1. Clip negatives to 0 before log transform
    # 2. Drop any residual non-finite values (NaN / inf)
    clean = np.log1p(df[col].clip(lower=0).dropna())
    clean = clean[np.isfinite(clean)]

    ax.hist(clean, bins=50, color='steelblue', edgecolor='white')
    ax.set_title(f'log1p({col})')
    ax.set_xlabel('log1p value')
    ax.set_ylabel('count')

plt.suptitle('Outlier Distributions (log-scaled)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## RECOMMENDER SYSTEM DIAGNOSTICS

In [ ]:
# ── Filter to valid interactions only ─────────────────────
rec_df = df.dropna(subset=['buyer_id','product_id','rating']).copy()
rec_df = rec_df[rec_df['rating'].between(1,5)]

# Remove duplicate (user, item) — keep highest-rated or most recent
rec_df = rec_df.sort_values('review_date', ascending=False)
rec_df = rec_df.drop_duplicates(subset=['buyer_id','product_id'], keep='first')

n_users   = rec_df['buyer_id'].nunique()
n_items   = rec_df['product_id'].nunique()
n_inter   = len(rec_df)
sparsity  = 1 - (n_inter / (n_users * n_items))

print("=" * 65)
print("3.1 INTERACTION MATRIX SUMMARY")
print("=" * 65)
print(f"  Unique users          : {n_users:>12,}")
print(f"  Unique products       : {n_items:>12,}")
print(f"  Total interactions    : {n_inter:>12,}")
print(f"  Matrix size           : {n_users:,} × {n_items:,} = {n_users*n_items:,}")
print(f"  SPARSITY              : {sparsity:.6f}  ({sparsity*100:.4f}%)")
print(f"  Density               : {(1-sparsity)*100:.4f}%")

# ── Per-user & per-item distributions ─────────────────────
user_counts = rec_df.groupby('buyer_id').size()
item_counts = rec_df.groupby('product_id').size()

print("\n" + "=" * 65)
print("3.2 INTERACTION DISTRIBUTION TABLES")
print("=" * 65)

print("\n  USER activity (reviews per user):")
user_dist = pd.cut(user_counts, bins=[0,1,2,5,10,20,50,100,np.inf],
                   labels=['1','2','3-5','6-10','11-20','21-50','51-100','100+'])
user_dist_df = user_dist.value_counts().sort_index().reset_index()
user_dist_df.columns = ['interactions_range','user_count']
user_dist_df['pct_users'] = (user_dist_df['user_count'] / n_users * 100).round(2)
user_dist_df['cumulative_%'] = user_dist_df['pct_users'].cumsum().round(2)
print(user_dist_df.to_string(index=False))

print("\n  ITEM review counts (reviews per product):")
item_dist = pd.cut(item_counts, bins=[0,1,2,5,10,20,50,100,np.inf],
                   labels=['1','2','3-5','6-10','11-20','21-50','51-100','100+'])
item_dist_df = item_dist.value_counts().sort_index().reset_index()
item_dist_df.columns = ['reviews_range','item_count']
item_dist_df['pct_items'] = (item_dist_df['item_count'] / n_items * 100).round(2)
print(item_dist_df.to_string(index=False))

# ── Cold-start severity ────────────────────────────────────
print("\n" + "=" * 65)
print("3.3 COLD-START SEVERITY REPORT")
print("=" * 65)

thresholds = [1, 2, 3, 5, 10]
print("\n  USERS:")
for t in thresholds:
    pct = (user_counts <= t).mean() * 100
    print(f"    ≤ {t:>2} interactions : {pct:.2f}% of users")

print("\n  ITEMS:")
for t in thresholds:
    pct = (item_counts <= t).mean() * 100
    print(f"    ≤ {t:>2} reviews      : {pct:.2f}% of items")

power_users = (user_counts >= 20).mean() * 100
print(f"\n  Power users (≥ 20 reviews): {power_users:.2f}%")

# ── Long-tail & activity plots ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Item long-tail
item_sorted = item_counts.sort_values(ascending=False).reset_index(drop=True)
axes[0].plot(item_sorted.values, color='crimson', linewidth=1.2)
axes[0].fill_between(range(len(item_sorted)), item_sorted.values, alpha=0.15, color='crimson')
axes[0].set_yscale('log'); axes[0].set_xscale('log')
axes[0].set_title('Item Long-Tail Distribution (log-log)', fontweight='bold')
axes[0].set_xlabel('Item rank'); axes[0].set_ylabel('# Reviews (log)')
axes[0].axvline(x=int(n_items*0.2), color='orange', ls='--', label='Top 20% items')
axes[0].legend()

# User activity histogram
axes[1].hist(np.log1p(user_counts.values), bins=60, color='steelblue', edgecolor='white')
axes[1].set_title('User Activity Distribution (log1p scale)', fontweight='bold')
axes[1].set_xlabel('log1p(# reviews per user)'); axes[1].set_ylabel('# Users')

plt.suptitle('Recommender System: Sparsity & Long-Tail Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## RATING & FEEDBACK ANALYSIS

In [ ]:
valid_ratings = df.dropna(subset=['rating'])
valid_ratings = valid_ratings[valid_ratings['rating'].between(1,5)]

print("=" * 65)
print("4.1 RATING DISTRIBUTION")
print("=" * 65)

rating_dist = valid_ratings['rating'].value_counts().sort_index()
rating_pct  = (rating_dist / len(valid_ratings) * 100).round(2)
rating_table = pd.DataFrame({'count': rating_dist, 'percentage': rating_pct})
print(rating_table)

from scipy.stats import skew, kurtosis
sk = skew(valid_ratings['rating'].dropna())
ku = kurtosis(valid_ratings['rating'].dropna())
mean_r = valid_ratings['rating'].mean()
print(f"\n  Mean rating   : {mean_r:.3f}")
print(f"  Skewness      : {sk:.3f}  {'⚠ LEFT-skewed (positivity bias)' if sk < -0.5 else '→ roughly symmetric'}")
print(f"  Kurtosis      : {ku:.3f}")
print(f"  5-star share  : {rating_pct.get(5.0, 0):.1f}%  ← watch for J-curve bias")

# ── Rating vs Likes/Dislikes ───────────────────────────────
print("\n" + "=" * 65)
print("4.2 RATING × LIKES/DISLIKES")
print("=" * 65)

feedback = valid_ratings.groupby('rating').agg(
    avg_likes    = ('likes', 'mean'),
    median_likes = ('likes', 'median'),
    avg_dislikes = ('dislikes', 'mean'),
    review_count = ('rating', 'count')
).round(3)
print(feedback)

# ── Rating vs Review Length ────────────────────────────────
print("\n" + "=" * 65)
print("4.3 RATING × REVIEW LENGTH")
print("=" * 65)

valid_ratings = valid_ratings.copy()
valid_ratings['text_len_chars'] = valid_ratings['review_text'].fillna('').str.len()
valid_ratings['text_len_words'] = valid_ratings['review_text'].fillna('').str.split().str.len()

length_by_rating = valid_ratings.groupby('rating').agg(
    avg_chars  = ('text_len_chars', 'mean'),
    avg_words  = ('text_len_words', 'mean'),
    pct_empty  = ('text_len_chars', lambda x: (x == 0).mean() * 100)
).round(2)
print(length_by_rating)

# ── Visualizations ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(rating_dist.index.astype(str), rating_dist.values,
            color=['#d32f2f','#f57c00','#fbc02d','#388e3c','#1565c0'])
axes[0].set_title('Rating Distribution', fontweight='bold')
axes[0].set_xlabel('Stars'); axes[0].set_ylabel('Count')
for i, (v, p) in enumerate(zip(rating_dist.values, rating_pct.values)):
    axes[0].text(i, v + max(rating_dist)*0.01, f'{p:.1f}%', ha='center', fontsize=9)

valid_ratings.boxplot(column='text_len_words', by='rating', ax=axes[1])
axes[1].set_title('Review Length vs Rating', fontweight='bold')
axes[1].set_xlabel('Rating'); axes[1].set_ylabel('Word count')

feedback['avg_likes'].plot(kind='bar', ax=axes[2], color='steelblue')
axes[2].set_title('Avg Likes by Rating', fontweight='bold')
axes[2].set_xlabel('Rating'); axes[2].set_ylabel('Avg likes')

plt.suptitle('Rating & Feedback Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## TEMPORAL ANALYSIS

In [ ]:
temp_df = df.dropna(subset=['review_date']).copy()
temp_df['year_month'] = temp_df['review_date'].dt.to_period('M')
temp_df['year']       = temp_df['review_date'].dt.year

print("=" * 65)
print("5.1 REVIEWS PER YEAR")
print("=" * 65)
print(temp_df['year'].value_counts().sort_index().to_string())

# ── Monthly trend ──────────────────────────────────────────
monthly = temp_df.groupby('year_month').size().reset_index()
monthly.columns = ['period','count']
monthly['period_dt'] = monthly['period'].dt.to_timestamp()

print("\n" + "=" * 65)
print("5.2 REVIEW DATE vs DATE BOUGHT (LATENCY)")
print("=" * 65)

latency_df = df.dropna(subset=['review_date','date_bought']).copy()
latency_df['latency_days'] = (latency_df['review_date'] - latency_df['date_bought']).dt.days
latency_df = latency_df[latency_df['latency_days'].between(0, 365)]  # reasonable window

print(latency_df['latency_days'].describe().round(2))
print(f"\n  % reviewed within 7 days  : {(latency_df['latency_days'] <= 7).mean()*100:.2f}%")
print(f"  % reviewed within 30 days : {(latency_df['latency_days'] <= 30).mean()*100:.2f}%")
print(f"  % reviewed within 90 days : {(latency_df['latency_days'] <= 90).mean()*100:.2f}%")
print(f"  Negative latency (anomaly) : {(latency_df['latency_days'] < 0).sum():,} rows")

# ── Plots ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(monthly['period_dt'], monthly['count'], color='teal', linewidth=1.5)
axes[0].fill_between(monthly['period_dt'], monthly['count'], alpha=0.2, color='teal')
axes[0].set_title('Monthly Review Volume', fontweight='bold')
axes[0].set_xlabel('Date'); axes[0].set_ylabel('# Reviews')
axes[0].tick_params(axis='x', rotation=45)

axes[1].hist(latency_df['latency_days'], bins=60, color='coral', edgecolor='white')
axes[1].set_title('Review Latency: Days After Purchase', fontweight='bold')
axes[1].set_xlabel('Days'); axes[1].set_ylabel('Count')
axes[1].axvline(latency_df['latency_days'].median(), color='black', ls='--', label='Median')
axes[1].legend()

plt.suptitle('Temporal Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## NLP / TEXT ANALYSIS

In [ ]:
# ── 6.1 Language detection heuristic ──────────────────────

BENGALI_RANGE = re.compile(r'[\u0980-\u09FF]')  # Unicode Bengali block

COMMON_BANGLISH = [
    'valo','bhalo','khub','boro','choto','sundor','dami','kharap',
    'onek','kono','ami','apni','ata','eta','sob','kintu','tobe',
    'theke','holo','gelo','ache','nei','na','haan','jodi','mane',
    'lagche','laglo','nibo','dilam','pelam','bhalo','valo',
]
BANGLISH_PATTERN = re.compile(
    r'\b(' + '|'.join(BANGLISH_COMMON := COMMON_BANGLISH) + r')\b', re.IGNORECASE
)
ENGLISH_PATTERN = re.compile(r'[a-zA-Z]{2,}')

def detect_language(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return 'empty'
    has_bengali  = bool(BENGALI_RANGE.search(text))
    has_banglish = bool(BANGLISH_PATTERN.search(text))
    has_english  = bool(ENGLISH_PATTERN.search(text))

    if has_bengali and has_english:
        return 'mixed'
    elif has_bengali:
        return 'bengali'
    elif has_banglish:
        return 'banglish'
    elif has_english:
        return 'english'
    else:
        return 'other'

print("=" * 65)
print("6.1 LANGUAGE DISTRIBUTION (heuristic)")
print("=" * 65)

# Sample for speed if needed; remove [:500000] for full pass
sample_df = df.dropna(subset=['review_text']).copy()
sample_df['lang'] = sample_df['review_text'].apply(detect_language)

lang_dist = sample_df['lang'].value_counts()
lang_pct  = (lang_dist / len(sample_df) * 100).round(2)
lang_table= pd.DataFrame({'count': lang_dist, 'pct': lang_pct})
print(lang_table)

# ── 6.2 Text statistics ────────────────────────────────────
print("\n" + "=" * 65)
print("6.2 TEXT STATISTICS")
print("=" * 65)

sample_df['char_len']  = sample_df['review_text'].str.len()
sample_df['word_len']  = sample_df['review_text'].str.split().str.len()
sample_df['has_emoji'] = sample_df['review_text'].apply(
    lambda t: bool(re.search(r'[\U00010000-\U0010ffff]|[\u2600-\u27BF]', str(t)))
)

print(sample_df[['char_len','word_len']].describe().round(2))
print(f"\n  % reviews with emojis    : {sample_df['has_emoji'].mean()*100:.2f}%")
print(f"  % reviews empty/< 5 chars: {(sample_df['char_len'] < 5).mean()*100:.2f}%")

# Vocab size (token-level)
from itertools import islice
all_tokens = []
for text in islice(sample_df['review_text'].dropna(), 200_000):  # cap for speed
    all_tokens.extend(str(text).lower().split())

token_freq = Counter(all_tokens)
vocab_size  = len(token_freq)
hapax       = sum(1 for v in token_freq.values() if v == 1)  # words appearing once

print(f"\n  Approx vocab size (200k sample)  : {vocab_size:,}")
print(f"  Hapax legomena (freq=1)          : {hapax:,}  ({hapax/vocab_size*100:.1f}%)")
print(f"  Top-50 token coverage            : "
      f"{sum(v for _,v in token_freq.most_common(50))/len(all_tokens)*100:.2f}%")

# ── 6.3 Banglish spelling variability (code-mixing risk) ───
print("\n" + "=" * 65)
print("6.3 BANGLISH SPELLING VARIABILITY EXAMPLES")
print("=" * 65)

VARIANT_GROUPS = {
    'good'   : ['valo','bhalo','balo','valoo','bhalo','vhalo'],
    'very'   : ['khub','kub','khob','khub'],
    'product': ['product','prdouct','prduct','prodct'],
    'nice'   : ['sundor','sundar','shundor','shundar','shundhor'],
    'bad'    : ['kharap','kharAp','khrap','kharáp'],
}

for concept, variants in VARIANT_GROUPS.items():
    total = sum(token_freq.get(v, 0) for v in variants)
    breakdown = {v: token_freq.get(v, 0) for v in variants if token_freq.get(v, 0) > 0}
    print(f"  '{concept}': {breakdown}  →  total: {total:,}")

# ── 6.4 Noisy token flags ──────────────────────────────────
print("\n" + "=" * 65)
print("6.4 NOISE & PREPROCESSING RISKS")
print("=" * 65)

# Repeated characters: "niceeeee", "khubbbbb"
repeat_char = re.compile(r'(.)\1{3,}')  # same char 4+ times
noisy_df = sample_df[sample_df['review_text'].str.contains(
    r'(.)\1{3,}', regex=True, na=False)]
print(f"  Reviews with char repetition (niceeeee): {len(noisy_df):,} ({len(noisy_df)/len(sample_df)*100:.2f}%)")

# All-digit tokens
all_digit_tokens = sum(1 for t in token_freq if t.isdigit())
print(f"  All-digit tokens in vocab           : {all_digit_tokens:,}")

# Mixed-script tokens (Banglish collision)
mixed_script = sum(1 for t in token_freq if BENGALI_RANGE.search(t) and ENGLISH_PATTERN.search(t))
print(f"  Mixed-script tokens                 : {mixed_script:,}  ← tokenizer risk")

# Top noisy tokens
print(f"\n  Top 20 most frequent tokens:")
print("  " + str([t for t,c in token_freq.most_common(20)]))

# ── Plots ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Lang distribution
lang_colors = ['#3f51b5','#f44336','#ff9800','#4caf50','#9e9e9e']
axes[0].pie(lang_pct.values, labels=lang_pct.index,
            autopct='%1.1f%%', colors=lang_colors[:len(lang_pct)])
axes[0].set_title('Language Distribution', fontweight='bold')

# Word length histogram
axes[1].hist(sample_df['word_len'].clip(0,100), bins=50, color='mediumorchid', edgecolor='white')
axes[1].set_title('Review Length (words)', fontweight='bold')
axes[1].set_xlabel('Word count'); axes[1].set_ylabel('# Reviews')
axes[1].axvline(sample_df['word_len'].median(), color='red', ls='--', label='Median')
axes[1].legend()

# Top 30 tokens (log scale)
top30 = token_freq.most_common(30)
axes[2].barh([t for t,_ in reversed(top30)],
             [np.log1p(c) for _,c in reversed(top30)], color='teal')
axes[2].set_title('Top 30 Tokens (log freq)', fontweight='bold')
axes[2].set_xlabel('log1p(frequency)')

plt.suptitle('NLP Text Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## IMAGE & METADATA INSIGHTS

In [ ]:
print("=" * 65)
print("7.1 IMAGE COVERAGE")
print("=" * 65)

review_df = df.dropna(subset=['buyer_id'])  # exclude product-only rows

pct_with_images = review_df['has_images'].mean() * 100
print(f"  % reviews with images     : {pct_with_images:.2f}%")
print(f"  Avg images per review     : {review_df['image_count'].mean():.3f}")
print(f"  Max images in a review    : {review_df['image_count'].max()}")

print("\n  IMAGE vs NO IMAGE (avg metrics):")
img_impact = review_df.groupby('has_images').agg(
    avg_rating   = ('rating', 'mean'),
    avg_likes    = ('likes', 'mean'),
    avg_dislikes = ('dislikes', 'mean'),
    count        = ('rating', 'count')
).round(3)
print(img_impact)

print("\n" + "=" * 65)
print("7.2 SELLER REPLY ANALYSIS")
print("=" * 65)

pct_replied = review_df['has_reply'].mean() * 100
print(f"  % reviews with seller reply: {pct_replied:.2f}%")

reply_impact = review_df.groupby('has_reply').agg(
    avg_rating   = ('rating', 'mean'),
    avg_likes    = ('likes', 'mean'),
    avg_dislikes = ('dislikes', 'mean'),
    count        = ('has_reply', 'count')
).round(3)
print(reply_impact)

# ── Does seller reply correlate with low-rating mitigation?
low_rating = review_df[review_df['rating'] <= 2]
reply_on_low = low_rating['has_reply'].mean() * 100
print(f"\n  Seller reply rate on 1–2 star reviews: {reply_on_low:.2f}%")
print("  → High reply rate on negatives = seller reputation management signal")

## CATEGORY ANALYSIS

In [ ]:
cat_df = df.dropna(subset=['category_l1'])

print("=" * 65)
print("8.1 TOP-LEVEL CATEGORY DISTRIBUTION")
print("=" * 65)

cat_dist = cat_df['category_l1'].value_counts()
cat_pct  = (cat_dist / len(cat_df) * 100).round(2)
cat_table= pd.DataFrame({'review_count': cat_dist, 'pct': cat_pct})
print(cat_table.head(20))

print("\n" + "=" * 65)
print("8.2 AVERAGE RATING PER CATEGORY")
print("=" * 65)

cat_rating = cat_df.groupby('category_l1').agg(
    avg_rating     = ('rating', 'mean'),
    review_count   = ('rating', 'count'),
    pct_5star      = ('rating', lambda x: (x==5).mean()*100),
    pct_1star      = ('rating', lambda x: (x==1).mean()*100),
).round(3).sort_values('review_count', ascending=False)
print(cat_rating.head(20))

# Imbalance index: ratio of largest to median category
imbalance = cat_dist.iloc[0] / cat_dist.median()
print(f"\n  Category imbalance ratio (top/median): {imbalance:.1f}x")
print("  ⚠ High imbalance degrades category-aware rec-sys models" if imbalance > 10 else "  ✓ Moderate imbalance")

# ── Plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

top15 = cat_dist.head(15)
axes[0].barh(top15.index[::-1], top15.values[::-1], color='royalblue')
axes[0].set_title('Top 15 Categories by Review Count', fontweight='bold')
axes[0].set_xlabel('# Reviews')

top15_rating = cat_rating.head(15)['avg_rating'].sort_values()
axes[1].barh(top15_rating.index, top15_rating.values,
             color=['#d32f2f' if v < 3.5 else '#4caf50' for v in top15_rating.values])
axes[1].axvline(3.5, color='black', ls='--', linewidth=0.8, label='3.5 threshold')
axes[1].set_title('Avg Rating per Category (Top 15)', fontweight='bold')
axes[1].set_xlabel('Avg Rating'); axes[1].legend()

plt.suptitle('Category Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## Summary of the Dataset

In [ ]:
summary = {
    'Total Reviews'             : len(df),
    'Unique Users'              : df['buyer_id'].nunique(),
    'Unique Products'           : df['product_id'].nunique(),
    'Sparsity'                  : f"{sparsity*100:.4f}%",
    'Cold-start users (≤2 ints)': f"see Section 3 table",
    'Avg Rating'                : f"{df['rating'].mean():.3f}",
    'Rating Skew'               : f"{skew(df['rating'].dropna()):.3f}",
    '% Empty Reviews'           : f"{(df['review_text'].fillna('').str.len()<5).mean()*100:.2f}%",
    '% Reviews w/ Images'       : f"{df['has_images'].mean()*100:.2f}%",
    '% Reviews w/ Reply'        : f"{df['has_reply'].mean()*100:.2f}%",
    'Approx Vocab Size'         : f"{vocab_size:,}",
    'Hapax Ratio'               : f"{hapax/vocab_size*100:.1f}%",
    '% Emoji Reviews'           : f"{sample_df['has_emoji'].mean()*100:.2f}%",
}

print("\n" + "=" * 50)
print("       BANGLISHREV EDA — FINAL SUMMARY")
print("=" * 50)
for k, v in summary.items():
    print(f"  {k:<35}: {v}")
print("=" * 50)